In [ ]:
# Ensure that we are using the correct host
import socket
try:
    assert "gpu" in socket.gethostname()
    print(f"Running on {socket.gethostname()}. All is good!")
except:
    raise RuntimeError(f"Be sure to run on GPU! You are currently running on {socket.gethostname()}")

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

### Get data

In [ ]:
BASE_PATH = r"/shared/healthinfolab/datasets/ABCD/Irritability/Clinical_Data/Irritability/Release_5.0"
CSV_TYPE = "wide"

time_labels = ["0m", "12m", "24m", "36m", "48m"]

files = [
    os.path.join(BASE_PATH, f"abcd_cbcl_irr_index_release5.0_{t}_{CSV_TYPE}.csv")
    for t in time_labels
]

dfs = []
for i, (f, tlabel) in enumerate(zip(files, time_labels)):
    df = pd.read_csv(f)

    # Find the irritability column for that wave
    irr_col = f"cbcl_irr_index_cnst_{tlabel}"

    if irr_col not in df.columns:
        raise ValueError(f"{irr_col} not found in {f}")

    df = df[["src_subject_id", irr_col]].copy()
    df = df.rename(columns={irr_col: "cbcl_irr"})
    df["time"] = i

    dfs.append(df)

long_df = pd.concat(dfs, ignore_index=True)

print("Combined shape:", long_df.shape)
print(long_df)


In [ ]:
data_label = "cbcl_irr"

### Clean and filter data

In [ ]:
# Keep only necessary columns
long_df = long_df[["src_subject_id", data_label, "time"]]

# Drop missing irritability scores
long_df = long_df.dropna(subset=[data_label])

# Keep subjects with at least 2 timepoints
counts = long_df.groupby("src_subject_id")["time"].nunique()
valid_ids = counts[counts >= 2].index
long_df = long_df[long_df["src_subject_id"].isin(valid_ids)]

print("Subjects with >=2 waves:", len(valid_ids))

### Fit trajectories

In [ ]:
betas = []

for subject, group in long_df.groupby("src_subject_id"):
    group = group.sort_values("time")
    X_time = group["time"].values.reshape(-1, 1)
    y = group[data_label].values

    if len(y) >= 2:
        model = LinearRegression().fit(X_time, y)
        betas.append([
            subject,
            model.intercept_,
            model.coef_[0]
        ])

beta_df = pd.DataFrame(betas, columns=["id", "intercept", "slope"])

print("Trajectory parameters computed:", len(beta_df))

### Standardize parameters

In [ ]:
scaler = StandardScaler()
X_beta = scaler.fit_transform(beta_df[["intercept", "slope"]])

### Sleect number of classes (BIC)

In [ ]:
bics = []
models = []
K_range = range(1, 6)

for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=10)
    gmm.fit(X_beta)
    bics.append(gmm.bic(X_beta))
    models.append(gmm)

best_k = K_range[np.argmin(bics)]
best_model = models[np.argmin(bics)]

print("Best number of classes (BIC):", best_k)

### Assign classes and merge labels back

In [ ]:
beta_df["class"] = best_model.predict(X_beta)
beta_df["class_prob"] = best_model.predict_proba(X_beta).max(axis=1)

print(beta_df["class"].value_counts())

long_df = long_df.merge(
    beta_df[["id", "class"]],
    left_on="src_subject_id",
    right_on="id",
    how="left"
)

beta_df.to_csv("/home/rif17002/honors_thesis/Clinical/trajectory_analysis_output/trajectory_classes.csv", index=False)

# Average Probability
print("\nAverage Class Probability:", beta_df["class_prob"].mean())

### Plot trajectory classes

In [ ]:
output_dir = "/home/rif17002/honors_thesis/Clinical/trajectory_analysis_output"
os.makedirs(output_dir, exist_ok=True)

plt.figure(figsize=(10, 7))

colors = plt.cm.tab10.colors

for idx, c in enumerate(sorted(beta_df["class"].unique())):
    ids = beta_df[beta_df["class"] == c]["id"]
    subset = long_df[long_df["src_subject_id"].isin(ids)]

    mean_traj = (
        subset.groupby("time")[data_label]
        .mean()
        .sort_index()
    )

    plt.plot(
        mean_traj.index,
        mean_traj.values,
        linewidth=3,
        label=f"Class {c}",
        color=colors[idx % len(colors)]
    )

plt.title("Latent Trajectory Classes (Mean Irritability Over Time)")
plt.xlabel("Time (0=Baseline, 4=48m)")
plt.ylabel("Irritability Score")
plt.legend(title="Trajectory Class")
plt.tight_layout()

plt.savefig(os.path.join(output_dir, "all_classes_mean_trajectories.png"))
plt.close()


In [ ]:
output_dir = "/home/rif17002/honors_thesis/Clinical/trajectory_analysis_output"
os.makedirs(output_dir, exist_ok=True)

for c in sorted(beta_df["class"].unique()):
    plt.figure(figsize=(10, 7))

    ids = beta_df[beta_df["class"] == c]["id"]
    subset = long_df[long_df["src_subject_id"].isin(ids)]

    # Plot all individual trajectories
    for subject_id, group in subset.groupby("src_subject_id"):
        group = group.sort_values("time")
        plt.plot(
            group["time"],
            group[data_label],
            alpha=0.1,   # adjust transparency if needed
            color=(0.1, 0.1, 0.1)
        )

    plt.title(f"Latent Trajectory Class {c} ({len(subset)} Subjects)")
    plt.xlabel("Time (0=Baseline, 4=48m)")
    plt.ylabel("Irritability Score")
    plt.tight_layout()

    plt.savefig(
        os.path.join(output_dir, f"class_{c}_all_trajectories.png")
    )
    plt.close()